# JOA R Residual Scale Sweep & Repack

기존 `scale=0.05` 제출이 JOA anchor 대비 리더보드에서 개선된 사실을 이용해, 모델을 재학습하지 않고 R 잔차 보정 강도만 재탐색.

- JOA anchor LB: `1126.8664003703`

- R residual scale 0.05 LB: `1127.5514185677`

- 관측 개선: `+0.6850181974`

2024 deployable OOF에서 scale 곡선과 pitcher-cluster bootstrap CI를 계산한 뒤, OOF 최적점과 0.05 사이로 축소한 보수적 scale을 선택합니다. 
이 실험은 모델, 피처, seed, anchor를 변경하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, shutil, subprocess, sys, time, zipfile
import numpy as np
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/LG_AIMERS')
BUILD_ROOT = DRIVE_ROOT / 'joa_r_residual_final' / 'build_v1'
OOF_PATH = BUILD_ROOT / 'deployable_validation_predictions_2024.npz'

zip_candidates = [
    DRIVE_ROOT / 'joa_r_residual_final' / 'submit_JOA_R_residual_multiseed005.zip',
    DRIVE_ROOT / 'joa_r_residual_final' / 'submit_JM_R_residual_multiseed005.zip',
    DRIVE_ROOT / 'submit_JOA_R_residual_multiseed005.zip',
    DRIVE_ROOT / 'submit_JM_R_residual_multiseed005.zip',
]
FINAL_ZIP = next((p for p in zip_candidates if p.exists()), None)
assert OOF_PATH.exists(), f'OOF 파일이 없습니다: {OOF_PATH}'
assert FINAL_ZIP is not None, '기존 residual 제출 ZIP을 LG_AIMERS 또는 joa_r_residual_final 아래에 올려주세요.'

OUTPUT_ROOT = DRIVE_ROOT / 'joa_r_scale_sweep'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('OOF:', OOF_PATH)
print('기존 ZIP:', FINAL_ZIP)
print('출력:', OUTPUT_ROOT)


## 1. OOF scale 곡선 계산

동일 correction에 scale만 곱하므로 Brier 개선 곡선은 거의 정확한 오목한 이차함수입니다. `0.000~0.150`을 0.005 간격으로 확인합니다.


In [ ]:
# This NPZ is our own trusted OOF artifact. row_id/game_type were saved as
# NumPy object arrays, so loading them requires allow_pickle=True.
with np.load(OOF_PATH, allow_pickle=True) as z:
    row_id = z['row_id']
    pitcher_id = z['pitcher_id'].astype(str)
    game_type = z['game_type'].astype(str)
    y = z['y'].astype(np.float64)
    anchor = z['p_anchor'].astype(np.float64)
    correction = z['correction'].astype(np.float64)

r_mask = game_type == 'R'
reference = float(y.mean() * (1.0 - y.mean()))

def bss(pred):
    return 100000.0 * (1.0 - np.mean((np.asarray(pred) - y) ** 2) / reference)

anchor_bss = bss(anchor)
scales = np.round(np.arange(0.0, 0.1501, 0.005), 3)
rows = []
for scale in scales:
    pred = anchor.copy()
    pred[r_mask] = np.clip(anchor[r_mask] + scale * correction[r_mask], 1e-6, 1 - 1e-6)
    value = bss(pred)
    rows.append({'scale': scale, 'bss': value, 'delta_bss': value - anchor_bss})

curve = pd.DataFrame(rows)
local_best = curve.loc[curve['delta_bss'].idxmax()].copy()
delta005 = float(curve.loc[np.isclose(curve.scale, 0.05), 'delta_bss'].iloc[0])
LB_ANCHOR = 1126.8664003703
LB_SCALE005 = 1127.5514185677
LB_DELTA005 = LB_SCALE005 - LB_ANCHOR
curve['lb_delta_shape_projection'] = curve['delta_bss'] / delta005 * LB_DELTA005
curve['lb_score_shape_projection'] = LB_ANCHOR + curve['lb_delta_shape_projection']
curve.to_csv(OUTPUT_ROOT / 'scale_curve_2024.csv', index=False)

print('anchor local BSS:', anchor_bss)
print('local scale=0.05 delta:', delta005)
print('official scale=0.05 delta:', LB_DELTA005)
print('local optimum:')
display(local_best.to_frame().T)
display(curve.sort_values('delta_bss', ascending=False).head(12))


## 2. 보수적 후보 선택 및 bootstrap

리더보드 관측은 한 점뿐이므로 OOF 최적 scale을 그대로 사용하지 않습니다. 
`0.05 + 50% × (OOF 최적점 - 0.05)`로 축소하고, 최대 0.075로 제한합니다.


In [ ]:
local_opt = float(local_best['scale'])
conservative = round(np.clip(0.05 + 0.5 * (local_opt - 0.05), 0.05, 0.075) / 0.005) * 0.005
aggressive = round(np.clip(local_opt, 0.05, 0.10) / 0.005) * 0.005
candidate_scales = sorted(set([float(conservative), float(aggressive)]))

def cluster_ci(scale, repetitions=3000, seed=260819):
    pred = anchor.copy()
    pred[r_mask] = np.clip(anchor[r_mask] + scale * correction[r_mask], 1e-6, 1 - 1e-6)
    gain = (anchor - y) ** 2 - (pred - y) ** 2
    tab = pd.DataFrame({'pitcher': pitcher_id, 'gain': gain}).groupby('pitcher').gain.agg(['sum', 'size'])
    sums = tab['sum'].to_numpy(float)
    sizes = tab['size'].to_numpy(float)
    rng = np.random.default_rng(seed)
    values = np.empty(repetitions)
    for start in range(0, repetitions, 64):
        count = min(64, repetitions - start)
        idx = rng.integers(0, len(tab), size=(count, len(tab)))
        values[start:start+count] = 100000.0 * sums[idx].sum(1) / sizes[idx].sum(1) / reference
    point = 100000.0 * gain.mean() / reference
    return point, np.quantile(values, .025), np.quantile(values, .975), np.mean(values > 0)

summary_rows = []
for scale in sorted(set([0.05] + candidate_scales)):
    point, low, high, prob = cluster_ci(scale)
    projected = LB_ANCHOR + point / delta005 * LB_DELTA005
    summary_rows.append({
        'scale': scale, 'local_delta_bss': point, 'ci_low': low, 'ci_high': high,
        'positive_probability': prob, 'projected_lb_shape_only': projected,
    })
scale_summary = pd.DataFrame(summary_rows)
scale_summary.to_csv(OUTPUT_ROOT / 'scale_candidate_summary.csv', index=False)
print('보수 후보:', conservative, '/ OOF 최적 후보:', aggressive)
display(scale_summary)


## 3. 제출 ZIP 재패키징

기존 ZIP의 모델 파일은 그대로 복사하고 manifest의 `scale`만 변경합니다. `scale=0.05`보다 OOF가 낮은 후보는 만들지 않습니다.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def repack_scale(source, destination, scale):
    tmp = Path('/content') / f'repack_scale_{int(round(scale * 1000)):03d}.zip'
    if tmp.exists(): tmp.unlink()
    found = 0
    with zipfile.ZipFile(source, 'r') as zin, zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED, compresslevel=4) as zout:
        for info in zin.infolist():
            data = zin.read(info.filename)
            if info.filename.endswith('model/r_residual_manifest.json'):
                manifest = json.loads(data.decode('utf-8'))
                assert abs(float(manifest['scale']) - 0.05) < 1e-12, manifest['scale']
                manifest['scale'] = float(scale)
                manifest['version'] = str(manifest.get('version', '')) + f'-scale{int(round(scale*1000)):03d}'
                manifest['scale_tuning_note'] = 'same models/features; scale-only post validation tuning'
                data = json.dumps(manifest, ensure_ascii=False, indent=2).encode('utf-8')
                found += 1
            zout.writestr(info, data)
    assert found == 1, f'manifest found={found}'
    shutil.copy2(tmp, destination)
    tmp.unlink()
    with zipfile.ZipFile(destination) as z:
        assert z.testzip() is None
        names = set(z.namelist())
        assert 'script.py' in names and 'requirements.txt' in names
        assert not any(n.startswith('data/') or n.startswith('output/') for n in names)
    return {'path': str(destination), 'scale': scale, 'size_mib': destination.stat().st_size/1024**2,
            'sha256': sha256(destination), 'files': len(names)}

base_delta = float(scale_summary.loc[np.isclose(scale_summary.scale, .05), 'local_delta_bss'].iloc[0])
packages = []
for scale in candidate_scales:
    candidate_delta = float(scale_summary.loc[np.isclose(scale_summary.scale, scale), 'local_delta_bss'].iloc[0])
    if scale == 0.05 or candidate_delta <= base_delta:
        print('생성 생략:', scale, 'OOF가 scale=.05 이하')
        continue
    label = int(round(scale * 1000))
    destination = OUTPUT_ROOT / f'submit_JM_R_residual_scale{label:03d}.zip'
    packages.append(repack_scale(FINAL_ZIP, destination, scale))

package_manifest = pd.DataFrame(packages)
package_manifest.to_csv(OUTPUT_ROOT / 'submission_manifest.csv', index=False)
display(package_manifest)
assert packages, 'scale=.05보다 나은 OOF 후보가 없어 새 ZIP을 생성하지 않았습니다.'


## 4. 최종 선택

`scale_candidate_summary.csv`에서 다음 순서로 고릅니다.

1. CI 하한이 가장 안정적인 후보

2. 보수 후보와 공격 후보의 projected LB 차이가 0.2 미만이면 보수 후보

3. 한 번만 제출한다면 기본적으로 `conservative` ZIP

`projected_lb_shape_only`는 실제 점수 보장이 아니라 OOF 곡선의 모양을 공식 개선폭에 맞춰 본 참고치입니다.


In [ ]:
if packages:
    safe_path = OUTPUT_ROOT / f'submit_JM_R_residual_scale{int(round(conservative*1000)):03d}.zip'
    if not safe_path.exists():
        safe_path = Path(packages[0]['path'])
    print('권장 제출 후보:', safe_path)
